# TCC · EDA de Marketing Mix Modeling com Google Meridian

**Dados simulados oficiais, foco GEO × TEMPO.** Não há estimação posterior, ROI ou recomendação de orçamento.

Execute de cima para baixo em runtime novo Python 3.12+ (CPU). A primeira execução instala dependências; isso pode levar alguns minutos. Se o Colab já tinha versões diferentes de NumPy/TensorFlow carregadas, reinicie a sessão após instalar e execute novamente.

O bootstrap clona o repositório público quando necessário, preserva instalações compatíveis e salva tudo na pasta do projeto. A fonte de dados e a API estão fixadas por commit/checksum. Para uma cópia histórica exata do projeto, configure `REPO_REF` com o commit da entrega. Os dados não possuem nomes de canais reais ou localização geográfica real.


In [1]:
# Execute esta célula primeiro. Em Colab novo, usa somente CPU.
import os, sys, subprocess, importlib.util, importlib.metadata
from pathlib import Path

REPO_URL = "https://github.com/thiagossilva17/MMM_Google_Meridian_Project.git"
REPO_REF = "main"  # Para repetir uma execução publicada, substitua pelo SHA do commit.
IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
cwd = Path.cwd()
candidates = [cwd, *cwd.parents]
PROJECT_ROOT = next((p for p in candidates if (p / "src/config.py").exists()), None)
if PROJECT_ROOT is None:
    PROJECT_ROOT = cwd / "MMM_Google_Meridian_Project"
    if not PROJECT_ROOT.exists():
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "checkout", REPO_REF], check=True)
    elif not (PROJECT_ROOT / "src/config.py").exists():
        raise RuntimeError("Diretório de destino existe, mas não contém o projeto esperado.")

required = ["meridian", "pandas", "numpy", "scipy", "matplotlib", "statsmodels", "tabulate"]
missing = any(importlib.util.find_spec(package) is None for package in required)
try:
    wrong_meridian = importlib.metadata.version("google-meridian") != "2.1.0"
except importlib.metadata.PackageNotFoundError:
    wrong_meridian = True
if missing or wrong_meridian:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(PROJECT_ROOT / "requirements-colab.txt")], check=True)

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / "outputs/.mplcache"))
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
from src.config import SEED, ROOT
from src.data import load_panel
from src.reporting import show
import numpy as np
np.random.seed(SEED)
geo_time_panel = load_panel()
print(f"Colab: {IN_COLAB}; projeto: {ROOT}; painel: {geo_time_panel.shape}")


Colab: False; projeto: /workspace/scratch/5d823022ed95/tcc-meridian; painel: (6240, 19)


## 00 — O que recebemos e podemos confiar na estrutura?

### Pergunta
O que recebemos e podemos confiar na estrutura?

### Motivação
Antes de procurar associações, é necessário verificar se cada linha representa um geo em uma semana, sem duplicação, lacunas ou valores impossíveis.

### Método
Verificar SHA-256, schema documentado no exemplo oficial, índice exportado, chave GEO × tempo, grade de 7 dias, nulos, domínios, população e pares gasto/impressões. Calcular CPMU com denominador zero indefinido.

### Interpretação antes de olhar os resultados
Painel completo permite as decomposições balanceadas posteriores. Controles negativos não são automaticamente inválidos: são índices sintéticos. Conversões não inteiras não devem ser arredondadas. Problemas de contrato interrompem a execução após salvar a auditoria.

**Dependências:** painel validado e funções em `src/eda.py`. O código reutilizável está comentado e pode ser aberto para inspecionar as fórmulas. Todos os resultados são calculados na execução, não preenchidos manualmente.


In [2]:
from src.eda import data_audit
report_00 = data_audit(geo_time_panel)
show(report_00)

### Resultado observado e conclusão parcial

Recebemos 6,240 registros, 40 geos e 156 semanas, de 2021-01-25 a 2024-01-15. A grade é semanal, completa e sem duplicatas, nulos ou violações de domínio verificadas. População é constante dentro de cada geo.

O CSV original tem uma coluna de índice exportado, validada como 0…N−1 e removida apenas da cópia processada. Conversões são valores contínuos simulados; controles podem ser negativos na escala fornecida. Há 5 canais pagos, uma mídia orgânica e Promo. Não há reach/frequency; nenhuma variável desse tipo foi inventada.

CPMU = gasto/impressões. 7496 razões são indefinidas por ausência de impressões e permanecem NaN. Conferir os contadores de inconsistências na auditoria. Isso distingue zero atividade de custo zero.

### O que ainda NÃO podemos concluir

As relações são descritivas. Não estimamos ROI, contribuição incremental ou efeitos causais.
Os dados são simulados: padrões não descrevem um mercado real e não validam, por si, a identificação de um MMM.

### CHECKPOINT

- **O que descobrimos?** As evidências numéricas acima e as tabelas exportadas respondem à pergunta deste capítulo.
- **Quais problemas apareceram?** Os limites e alertas estão descritos nos resultados; nenhuma observação foi excluída ou imputada.
- **O que falta investigar?** Distribuições, concentração de investimento, zeros e extremos.
- **Podemos avançar?** Podemos prosseguir com a investigação exploratória após revisar estes achados; isso não aprova automaticamente a modelagem.


**data_audit** — 29 linhas; CSV completo em `outputs/tables/`.

,metric,value
0,rows,6240
1,geos,40
2,periods,156
3,expected_rows,6240
4,unique_geo_time,6240
5,panel_completeness,1.0
6,duplicate_keys,0
7,missing_geo_time,0
8,missing_cells,0
9,nonfinite_numeric,0


**dictionary** — 19 linhas; CSV completo em `outputs/tables/`.

,variable,role,dtype,dimension,unit,missing,zeros,min,max,mean,std
0,geo,geo,object,geo × time,geo,0,NaN,Geo0,Geo9,NaN,NaN
1,time,time,datetime64[ns],geo × time,time,0,NaN,2021-01-25 00:00:00,2024-01-15 00:00:00,NaN,NaN
2,Channel0_impression,media,int64,geo × time,impressões,0,755.0,0,5192032,8.851635e+05,8.192795e+05
3,Channel1_impression,media,int64,geo × time,impressões,0,1742.0,0,4397540,5.206055e+05,6.205734e+05
4,Channel2_impression,media,int64,geo × time,impressões,0,4010.0,0,5610156,2.605050e+05,5.551397e+05
5,Channel3_impression,media,int64,geo × time,impressões,0,169.0,0,7635147,1.806810e+06,1.304714e+06
6,Channel4_impression,media,int64,geo × time,impressões,0,820.0,0,6975542,9.816780e+05,9.141036e+05
7,competitor_sales_control,control,float64,geo × time,escala sintética; unidade não especificada,0,0.0,-4.825634,3.924682,-4.441200e-02,1.210067e+00
8,sentiment_score_control,control,float64,geo × time,escala sintética; unidade não especificada,0,0.0,-4.230392,4.320259,-3.367510e-02,1.175478e+00
9,Channel0_spend,spend,float64,geo × time,unidade monetária não especificada,0,755.0,0.0,38071.73,6.490659e+03,6.007550e+03


**CPMU** — 5 linhas; CSV completo em `outputs/tables/`.

,channel,undefined,n,mean,median,std,variance,min,max,p01,p05,p25,p50,p75,p95,p99,iqr,cv,zeros,zero_share
0,Channel0,755,5485,0.007333,0.007333,2.433514e-10,5.921988e-20,0.007333,0.007333,0.007333,0.007333,0.007333,0.007333,0.007333,0.007333,0.007333,3.470033e-10,3.318704e-08,0,0.0
1,Channel1,1742,4498,0.009641,0.009641,3.226900e-10,1.041288e-19,0.009641,0.009641,0.009641,0.009641,0.009641,0.009641,0.009641,0.009641,0.009641,4.495688e-10,3.346979e-08,0,0.0
2,Channel2,4010,2230,0.007431,0.007431,2.582488e-10,6.669246e-20,0.007431,0.007431,0.007431,0.007431,0.007431,0.007431,0.007431,0.007431,0.007431,3.643462e-10,3.475326e-08,0,0.0
3,Channel3,169,6071,0.007793,0.007793,2.581763e-10,6.665502e-20,0.007793,0.007793,0.007793,0.007793,0.007793,0.007793,0.007793,0.007793,0.007793,3.653120e-10,3.312994e-08,0,0.0
4,Channel4,820,5420,0.007792,0.007792,2.578769e-10,6.650050e-20,0.007792,0.007792,0.007792,0.007792,0.007792,0.007792,0.007792,0.007792,0.007792,3.683969e-10,3.309546e-08,0,0.0


[Figura gerada; veja outputs/figures ou execute esta célula para exibir.]

[Figura gerada; veja outputs/figures ou execute esta célula para exibir.]